In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 6.5 MB/s eta 0:00:00


In [2]:
import os
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.cm as cm
from PIL import Image
from IPython.display import display
from collections import Counter


from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


## CFG

Единственная ячейка, которую нужно править.

In [3]:
class CFG:
    """Конфиг обучения и инференса."""

    # ── данные ──────────────────────────────────────────────────
    DATA_ROOT   = Path("/kaggle/input/datasets/antonoof/publicdatabuilding/dataset")
    TRAIN_DIR   = DATA_ROOT / "train"     # внутри images/ и labels/
    VAL_DIR     = DATA_ROOT / "val"
    PREDICT_DIR = None                    # None -> берём VAL_DIR/images

    CLASSES = [
        "Dump truck",         # 0
        "Excavator",          # 1
        "Bucket loader",      # 2   Motor grader + Big + Standard
        "Bulldozer",          # 3
        "Mixer",              # 4
        "Crane manipulator",  # 5
        "Autocran",           # 6
        "Forklift",           # 7
        "Drilling rig",       # 8   данных нет, место под дообучение
    ]

    # ── модель и обучение ───────────────────────────────────────
    MODEL     = "yolo26m.pt"
    EPOCHS    = 50  # 50 исправить
    IMGSZ     = 1024
    BATCH     = 16
    PATIENCE  = 20
    OPTIMIZER = "auto"        # auto | SGD | AdamW
    LR0       = 0.01
    COS_LR    = True
    WORKERS   = 2
    SEED      = 42
    DEVICE    = [0,1]             # 0 -> GPU, "cpu" -> CPU

    # аугментации
    HSV_H, HSV_S, HSV_V = 0.015, 0.7, 0.4
    FLIPUD, FLIPLR      = 0.0, 0.5
    DEGREES, TRANSLATE  = 0.0, 0.1
    SCALE, SHEAR        = 0.5, 0.0
    MOSAIC              = 1.0
    CLOSE_MOSAIC        = 10   # последние N эпох без мозаики

    # ── инференс ────────────────────────────────────────────────
    CONF      = 0.25
    IOU       = 0.6
    MAX_DET   = 100
    N_PREVIEW = 5              # сколько картинок показать

    # ── вывод ───────────────────────────────────────────────────
    PROJECT   = "/kaggle/working/runs"
    RUN_NAME  = "construction"
    YAML_PATH = Path("/kaggle/working/data.yaml")

    IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

    @classmethod
    def predict_dir(cls):
        return Path(cls.PREDICT_DIR) if cls.PREDICT_DIR else cls.VAL_DIR / "images"

    @classmethod
    def check(cls):
        """Проверяет структуру датасета и печатает сводку."""
        for split, d in (("train", cls.TRAIN_DIR), ("val", cls.VAL_DIR)):
            img_dir, lbl_dir = d / "images", d / "labels"
            assert img_dir.is_dir(), f"нет {img_dir}"
            assert lbl_dir.is_dir(), f"нет {lbl_dir} (ultralytics ищет labels рядом с images)"

            imgs = [p for p in img_dir.glob("*") if p.suffix.lower() in cls.IMG_EXTS]
            lbls = list(lbl_dir.glob("*.txt"))
            boxes, bad = 0, []
            for lbl in lbls:
                for i, line in enumerate(lbl.read_text().splitlines(), 1):
                    p = line.split()
                    if not p:
                        continue
                    if len(p) != 5 or not (0 <= int(p[0]) < len(cls.CLASSES)):
                        bad.append(f"{lbl.name}:{i}")
                    boxes += 1

            print(f"{split:<6} {len(imgs):>6} изображений  {len(lbls):>6} разметок  {boxes:>7} боксов")
            if len(imgs) != len(lbls):
                print(f"       ! images != labels, ultralytics пропустит лишние")
            if bad:
                print(f"       ! битых строк: {len(bad)}, первые: {bad[:5]}")

        print(f"\nклассов: {len(cls.CLASSES)}  |  предсказываем по: {cls.predict_dir()}")


CFG.check()

train    9254 изображений    9254 разметок    18576 боксов
val      2336 изображений    2336 разметок     3905 боксов

классов: 9  |  предсказываем по: /kaggle/input/datasets/antonoof/publicdatabuilding/dataset/val/images


## data.yaml

In [4]:
def build_yaml(cfg=CFG) -> Path:
    """Пишет data.yaml по конфигу и возвращает путь к нему."""
    text = (
        f"path: {cfg.DATA_ROOT}\n"
        f"train: {cfg.TRAIN_DIR.relative_to(cfg.DATA_ROOT)}/images\n"
        f"val: {cfg.VAL_DIR.relative_to(cfg.DATA_ROOT)}/images\n\n"
        f"nc: {len(cfg.CLASSES)}\n"
        "names:\n"
        + "".join(f"  {i}: {n}\n" for i, n in enumerate(cfg.CLASSES))
    )
    cfg.YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
    cfg.YAML_PATH.write_text(text)
    return cfg.YAML_PATH


print(build_yaml().read_text())

path: /kaggle/input/datasets/antonoof/publicdatabuilding/dataset
train: train/images
val: val/images

nc: 9
names:
  0: Dump truck
  1: Excavator
  2: Bucket loader
  3: Bulldozer
  4: Mixer
  5: Crane manipulator
  6: Autocran
  7: Forklift
  8: Drilling rig



## Обучение

In [5]:
model = YOLO(CFG.MODEL)

results = model.train(
    data=str(CFG.YAML_PATH),
    epochs=CFG.EPOCHS,
    imgsz=CFG.IMGSZ,
    batch=CFG.BATCH,
    patience=CFG.PATIENCE,
    optimizer=CFG.OPTIMIZER,
    lr0=CFG.LR0,
    cos_lr=CFG.COS_LR,
    workers=CFG.WORKERS,
    seed=CFG.SEED,
    device=CFG.DEVICE,
    project=CFG.PROJECT,
    name=CFG.RUN_NAME,
    exist_ok=True,
    # аугментации
    hsv_h=CFG.HSV_H, hsv_s=CFG.HSV_S, hsv_v=CFG.HSV_V,
    flipud=CFG.FLIPUD, fliplr=CFG.FLIPLR,
    degrees=CFG.DEGREES, translate=CFG.TRANSLATE,
    scale=CFG.SCALE, shear=CFG.SHEAR,
    mosaic=CFG.MOSAIC, close_mosaic=CFG.CLOSE_MOSAIC,
)

BEST = Path(results.save_dir) / "weights" / "best.pt"
print("\nлучший чекпоинт:", BEST)

Ultralytics 8.4.155 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937,

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       2/50      11.8G     0.8627      0.826   0.005355         10       1024: 100% ━━━━━━━━━━━━ 579/579 1.7it/s 5:46
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 73/73 2.9it/s 25.6s
                   all       2336       3905      0.555      0.447      0.477      0.343

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/50      11.8G     0.8529     0.6963   0.005382         13       1024: 100% ━━━━━━━━━━━━ 579/579 1.7it/s 5:45
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 73/73 2.8it/s 25.8s
                   all       2336       3905      0.529      0.443      0.464      0.323

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/50      11.8G     0.8313     0.6115   0.005066         13       1024: 100% ━━━━━━━━━

AttributeError: 'dict' object has no attribute 'save_dir'

## Метрики по классам

`Drilling rig` даст нули — объектов в датасете нет. Это ожидаемо, класс держит
место в голове модели под будущее дообучение.

In [ ]:
MIN_INSTANCES = 30   # ниже этого порога AP по классу нерепрезентативен

best_model = YOLO(BEST)
metrics = best_model.val(data=str(CFG.YAML_PATH), imgsz=CFG.IMGSZ, device=CFG.DEVICE, verbose=False)

# сколько объектов каждого класса в валидации
nt = getattr(metrics, "nt_per_class", None)
if nt is not None and len(nt) == len(CFG.CLASSES):
    val_counts = {i: int(n) for i, n in enumerate(nt)}
else:
    val_counts = Counter()
    for lbl in (CFG.VAL_DIR / "labels").glob("*.txt"):
        for line in lbl.read_text().splitlines():
            if line.strip():
                val_counts[int(line.split()[0])] += 1

pos = {int(c): k for k, c in enumerate(metrics.box.ap_class_index)}

rows = []
for i, name in enumerate(CFG.CLASSES):
    n = val_counts.get(i, 0)
    if i in pos:
        k = pos[i]
        rows.append({
            "id": i, "class": name, "n_val": n,
            "P": float(metrics.box.p[k]),
            "R": float(metrics.box.r[k]),
            "mAP50": float(metrics.box.ap50[k]),
            "mAP50-95": float(metrics.box.ap[k]),
            "status": "ok" if n >= MIN_INSTANCES else f"мало данных (<{MIN_INSTANCES})",
        })
    else:
        rows.append({
            "id": i, "class": name, "n_val": n,
            "P": np.nan, "R": np.nan, "mAP50": np.nan, "mAP50-95": np.nan,
            "status": "нет объектов в val",
        })

df = pd.DataFrame(rows)
ok = df[df["status"] == "ok"]

display(
    df.sort_values("mAP50", na_position="last").reset_index(drop=True)
      .style.format({c: "{:.3f}" for c in ["P", "R", "mAP50", "mAP50-95"]}, na_rep="—")
)

print(f"достоверных классов: {len(ok)} из {len(CFG.CLASSES)}, объектов в val: {int(ok['n_val'].sum())}")
if len(ok):
    w = ok["n_val"]
    print(f"  macro     mAP50 = {ok['mAP50'].mean():.3f}   mAP50-95 = {ok['mAP50-95'].mean():.3f}")
    print(f"  weighted  mAP50 = {np.average(ok['mAP50'], weights=w):.3f}   "
          f"mAP50-95 = {np.average(ok['mAP50-95'], weights=w):.3f}")
    print(f"  P = {ok['P'].mean():.3f}   R = {ok['R'].mean():.3f}")

bad = df[df["status"] != "ok"]
if len(bad):
    print("\nне учтены:")
    for _, r in bad.iterrows():
        print(f"  {r['id']} {r['class']:<20} n_val={r['n_val']:<5} {r['status']}")

print(f"\nultralytics по всем классам с объектами: "
      f"mAP50 = {metrics.box.map50:.3f}, mAP50-95 = {metrics.box.map:.3f}")

## Предсказание

Пять изображений выводятся по одному, каждое своим блоком — без сетки сабплотов,
так видно детали на полном разрешении.

In [ ]:
PALETTE = [tuple(int(c * 255) for c in cm.tab10(i % 10)[:3]) for i in range(len(CFG.CLASSES))]


def draw(img_bgr, boxes, classes, confs, cfg=CFG):
    """Рисует боксы с подписями. Толщина и шрифт масштабируются под размер картинки."""
    img = img_bgr.copy()
    h, w = img.shape[:2]
    thick = max(2, round(min(h, w) / 400))
    scale = max(0.5, min(h, w) / 1200)

    for (x1, y1, x2, y2), cls_id, conf in zip(boxes, classes, confs):
        color = PALETTE[int(cls_id)][::-1]          # tab10 в RGB -> BGR
        p1, p2 = (int(x1), int(y1)), (int(x2), int(y2))
        cv2.rectangle(img, p1, p2, color, thick)

        label = f"{cfg.CLASSES[int(cls_id)]} {conf:.2f}"
        (tw, th), base = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, scale, thick - 1)
        y_top = max(p1[1], th + base + 2)
        cv2.rectangle(img, (p1[0], y_top - th - base - 2), (p1[0] + tw + 4, y_top), color, -1)
        cv2.putText(img, label, (p1[0] + 2, y_top - base),
                    cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), thick - 1, cv2.LINE_AA)
    return img


def preview(model, cfg=CFG, seed=CFG.SEED):
    """Показывает N_PREVIEW случайных предсказаний, каждое отдельным изображением."""
    src = cfg.predict_dir()
    paths = [p for p in src.glob("*") if p.suffix.lower() in cfg.IMG_EXTS]
    assert paths, f"нет изображений в {src}"

    random.seed(seed)
    samples = random.sample(paths, min(cfg.N_PREVIEW, len(paths)))

    for n, path in enumerate(samples, 1):
        r = model.predict(path, conf=cfg.CONF, iou=cfg.IOU, max_det=cfg.MAX_DET,
                          device=cfg.DEVICE, verbose=False)[0]

        b = r.boxes
        img = draw(r.orig_img,
                   b.xyxy.cpu().numpy(),
                   b.cls.cpu().numpy().astype(int),
                   b.conf.cpu().numpy())

        found = ", ".join(f"{cfg.CLASSES[int(c)]} {f:.2f}"
                          for c, f in zip(b.cls.cpu().numpy(), b.conf.cpu().numpy())) or "ничего не найдено"
        print(f"[{n}/{len(samples)}] {path.name}  —  {found}")

        display(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
        print()


preview(best_model)